# Open HPD Violations — live SODA analysis

This notebook pulls **fresh** counts from the NYC Open Data SODA API
(`csn4-vhvf` / Open HPD Violations) — the same live source the React/Django
app uses (there is no local violations SQLite cache anymore).

### How to run (non-technical)
1. In Terminal, from the `notebooks/` folder:
   ```bash
   python3 -m venv .venv
   source .venv/bin/activate
   pip install -r requirements.txt
   ```
2. Make sure the project-root `.env` has `SOCRATA_APP_TOKEN=...`
3. Start Jupyter and open this file:
   ```bash
   jupyter notebook open_hpd_live_analysis.ipynb
   ```
4. Use **Run → Run All Cells** (first live queries can take 1–3 minutes).

No Django / frontend / AI service needed for this notebook.

In [ ]:
# Setup: load the live SODA helper next to this notebook.
# If this cell fails with ModuleNotFoundError, activate notebooks/.venv
# and run: pip install -r requirements.txt

from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import pandas as pd

from soda_live import fetch_live_overview, load_socrata_settings

pd.set_option("display.max_rows", 40)
pd.set_option("display.max_columns", 20)

cfg = load_socrata_settings()
print("Source: LIVE Socrata SODA API (not local SQLite)")
print(f"Dataset : {cfg['dataset_id']} @ {cfg['domain']}")
print(f"Env file: {cfg['env_path']}")
print(f"App token present: {cfg['has_app_token']}")
if not cfg["has_app_token"]:
    print("WARNING: add SOCRATA_APP_TOKEN to the root .env for better rate limits.")

## Optional SoQL filter

Leave `WHERE` empty for the full open-violations table.
Examples:
- `boro='BRONX'`
- `class='C'`
- `boro='BROOKLYN' AND class='C'`

Change the string below, then re-run the fetch cell.

In [ ]:
# Optional filter (SoQL WHERE). Empty string = all open violations.
WHERE = ""  # e.g. "boro='BRONX'"

print("Filter:", WHERE or "(none — full open table)")

## Fetch live aggregates

This cell talks to NYC Open Data. Expect it to take a while on ~3M rows.
If it times out, confirm `SOCRATA_APP_TOKEN` is set and try again.

In [ ]:
# Live pull — borough / class / status / monthly + a small sample of rows
overview = fetch_live_overview(where=WHERE or None)

print(f"Live matching rows: {overview['row_count']:,}")
print("Source metadata:", overview["settings"])

display(overview["by_boro"])
display(overview["by_class"])
display(overview["by_currentstatus"].head(15))
display(overview["by_month"].tail(12))

## Charts (from live API data)

In [ ]:
def plot_bars(df, title, horizontal=False):
    """Draw a bar chart from a name/value DataFrame."""
    if df is None or df.empty:
        print("No data for", title)
        return
    fig, ax = plt.subplots(figsize=(9, 4.5))
    if horizontal:
        ax.barh(df["name"], df["value"], color="#0b5fff")
        ax.invert_yaxis()
        ax.set_xlabel("Violations")
    else:
        ax.bar(df["name"], df["value"], color="#0b5fff")
        ax.set_ylabel("Violations")
        ax.tick_params(axis="x", rotation=30)
    ax.set_title(title)
    fig.tight_layout()
    plt.show()


plot_bars(overview["by_boro"], "Open violations by borough (live SODA)")
plot_bars(overview["by_class"], "Open violations by class (live SODA)")
plot_bars(overview["by_currentstatus"], "Top current statuses (live SODA)", horizontal=True)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(overview["by_month"]["name"], overview["by_month"]["value"], color="#0b5fff", linewidth=2)
ax.set_title("Inspections by month (live SODA)")
ax.set_ylabel("Inspections")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

## Sample rows (live)

Newest inspections from the API — useful to spot-check fields.
This is a small sample, not the full table.

In [ ]:
sample = overview["sample_rows"]
cols = [c for c in ["violationid", "boro", "class", "streetname", "housenumber", "zip", "currentstatus", "inspectiondate"] if c in sample.columns]
display(sample[cols] if cols else sample.head())

## Maintenance notes

| Task | What to do |
| --- | --- |
| Update API key | Edit root `.env` → `SOCRATA_APP_TOKEN`, restart kernel |
| Change dataset | Edit root `.env` → `SOCRATA_DATASET_ID` (keep `csn4-vhvf` for open only) |
| Prefer the web UI | Open the React app (live SODA list + charts + Refresh) |
| No Jupyter | `python run_live_analysis.py` in this folder |